# Lab 1 — Graphs in Python

**Discrete Mathematics — Graphs, Trees & Binary Trees**

In this lab you implement a small graph library from scratch.

**Representation.** A graph is a dictionary mapping each vertex to the **list of its neighbors** (the *adjacency list* representation from the lesson):

```python
G = {"A": ["B", "C"], "B": ["A"], "C": ["A"]}
```

We use `networkx` **only** to *draw* the graphs and to *check* your implementations — every algorithm must be written by hand.

**How to work.** Run the notebook from top to bottom. Replace each `# TODO` and remove the `raise NotImplementedError`. A printed ✔ means the exercise's tests pass.

**Contents**

1. Building a graph
2. Degrees and the handshake lemma
3. Breadth-first search (BFS)
4. Depth-first search (DFS) and connected components
5. Cycle detection
6. Bipartite graphs and 2-coloring
7. Greedy coloring
8. ★ Challenge — satisfiability (3-SAT) as a coloring problem

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque

def to_nx(G):
    """Convert a dict-of-lists graph into a networkx Graph (drawing/checking only)."""
    H = nx.Graph()
    H.add_nodes_from(G)
    for u in G:
        for v in G[u]:
            H.add_edge(u, v)
    return H

def draw_graph(G, coloring=None, pos=None, title=""):
    """Draw a dict-of-lists graph; `coloring` may map vertices to small integers."""
    H = to_nx(G)
    if pos is None:
        pos = nx.spring_layout(H, seed=7)
    palette = ["#e63946", "#2a9d8f", "#e9c46a", "#457b9d", "#f4a261", "#9b5de5"]
    if coloring is None:
        node_color = "#a8dadc"
    else:
        node_color = [palette[coloring[v] % len(palette)] for v in H.nodes()]
    plt.figure(figsize=(5.5, 4.2))
    nx.draw(H, pos, with_labels=True, node_color=node_color,
            node_size=650, font_weight="bold", edge_color="#666666")
    plt.title(title)
    plt.show()

print("Setup OK — networkx", nx.__version__)

## Exercise 1 — Building a graph

Write the three functions below.

1. `add_vertex(G, v)` — add vertex `v` to `G` (no effect if it is already there);
2. `add_edge(G, u, v)` — add the **undirected** edge $\{u, v\}$ (add missing endpoints; never duplicate a neighbor);
3. `build_graph(vertices, edges)` — build a fresh graph from a list of vertices and a list of pairs.

In [ ]:
def add_vertex(G, v):
    """Add vertex v to G if not already present."""
    # TODO
    raise NotImplementedError

def add_edge(G, u, v):
    """Add the undirected edge {u, v} to G (and its endpoints if needed)."""
    # TODO
    raise NotImplementedError

def build_graph(vertices, edges):
    """Return a new dict-of-lists graph with the given vertices and edges."""
    # TODO
    raise NotImplementedError

In [ ]:
EDGES = [("A","B"), ("A","C"), ("B","C"), ("B","D"), ("C","E"),
         ("D","E"), ("D","F"), ("E","F"), ("F","G")]
G1 = build_graph("ABCDEFG", EDGES)

assert set(G1) == set("ABCDEFG")
assert set(G1["A"]) == {"B", "C"} and set(G1["F"]) == {"D", "E", "G"}
add_edge(G1, "A", "B")                       # adding twice must not duplicate
assert G1["A"].count("B") == 1
print("Exercise 1 ✔")
draw_graph(G1, title="G1 — does it match the edge list?")

## Exercise 2 — Degrees and the handshake lemma

1. `degree(G, v)` — number of neighbors of `v`;
2. `degree_sequence(G)` — list of all degrees, sorted in decreasing order;
3. `num_edges(G)` — number of edges, **without** counting any edge twice.

Then the test cell checks the **handshake lemma** on `G1`:

$$\sum_{v \in V} \deg(v) = 2\,|E|.$$

**Question.** A graph has 9 vertices, all of degree 3. How many edges does it have?

In [ ]:
def degree(G, v):
    """Degree of vertex v in G."""
    # TODO
    raise NotImplementedError

def degree_sequence(G):
    """All degrees of G, sorted in decreasing order."""
    # TODO
    raise NotImplementedError

def num_edges(G):
    """Number of edges of G (hint: use the handshake lemma)."""
    # TODO
    raise NotImplementedError

In [ ]:
assert degree(G1, "F") == 3 and degree(G1, "G") == 1
assert degree_sequence(G1) == [3, 3, 3, 3, 3, 2, 1]
assert num_edges(G1) == len(EDGES)
assert sum(degree(G1, v) for v in G1) == 2 * num_edges(G1)   # handshake lemma
assert num_edges(G1) == to_nx(G1).number_of_edges()          # networkx agrees
print("Exercise 2 ✔  degree sequence:", degree_sequence(G1))

## Exercise 3 — Breadth-first search

Implement `bfs(G, s)`: return a dictionary `{v: d(s, v)}` giving the distance (number of edges) from `s` to every **reachable** vertex. Visit the vertices level by level using a queue (`collections.deque`, with `append` and `popleft`).

In [ ]:
def bfs(G, s):
    """Return {v: distance from s} for every vertex reachable from s."""
    # TODO — start from {s: 0} and a queue containing s
    raise NotImplementedError

In [ ]:
dist = bfs(G1, "A")
assert dist == nx.single_source_shortest_path_length(to_nx(G1), "A")
print("Exercise 3 ✔  distances from A:", dist)
draw_graph(G1, coloring=dist, title="G1 colored by distance from A (BFS layers)")

## Exercise 4 — Depth-first search and connected components

1. `dfs(G, s)` — return the **set** of vertices reachable from `s` (iterative version, with an explicit stack: a plain Python list with `append`/`pop`);
2. `connected_components(G)` — return the list of connected components, each one a set of vertices.

In [ ]:
def dfs(G, s):
    """Set of vertices reachable from s (iterative DFS with a stack)."""
    # TODO
    raise NotImplementedError

def connected_components(G):
    """List of the connected components of G (sets of vertices)."""
    # TODO — launch dfs from every not-yet-visited vertex
    raise NotImplementedError

In [ ]:
G2 = build_graph(range(1, 11),
                 [(1,2), (2,3), (3,1), (4,5), (5,6), (6,7), (7,4), (8,9)])
comps = connected_components(G2)
assert sorted(map(len, comps)) == [1, 2, 3, 4]
assert len(comps) == nx.number_connected_components(to_nx(G2))
assert dfs(G1, "A") == set(G1)            # G1 is connected
print("Exercise 4 ✔  components of G2:", comps)
draw_graph(G2, title="G2 — four connected components")

## Exercise 5 — Cycle detection

Implement `has_cycle(G)` for an **undirected** graph: run a DFS from every unvisited vertex, remembering each vertex's *parent* in the traversal; reaching an already-visited vertex that is **not** the parent reveals a cycle.

Recall from the lesson: a connected graph with no cycle is a **tree**, and then $|E| = |V| - 1$.

In [ ]:
def has_cycle(G):
    """True iff the undirected graph G contains a cycle."""
    # TODO — DFS keeping track of each vertex's parent
    raise NotImplementedError

In [ ]:
T = build_graph("ABCDEF", [("A","B"), ("A","C"), ("B","D"), ("B","E"), ("C","F")])
assert not has_cycle(T) and num_edges(T) == len(T) - 1     # a tree!
assert nx.is_tree(to_nx(T))                                # networkx agrees
assert has_cycle(G1) and has_cycle(G2)
P5 = build_graph(range(5), [(i, i + 1) for i in range(4)])
assert not has_cycle(P5)
print("Exercise 5 ✔")
draw_graph(T, title="T — connected and acyclic: a tree")

## Exercise 6 — Bipartite graphs and 2-coloring

A graph is **bipartite** when its vertices split into two sides such that every edge joins the two sides — equivalently, when it admits a proper **2-coloring** (adjacent vertices always get different colors).

Implement `two_color(G)`: return a dict `{vertex: 0 or 1}` describing a proper 2-coloring if one exists, and `None` otherwise. Color along a BFS, handling **every** component; finding two neighbors with the same color means an odd cycle is present.

In [ ]:
def two_color(G):
    """A proper 2-coloring of G as a dict, or None if G is not bipartite."""
    # TODO — BFS per component, alternating colors 0 / 1
    raise NotImplementedError

In [ ]:
C6  = build_graph(range(6), [(i, (i + 1) % 6) for i in range(6)])
C5  = build_graph(range(5), [(i, (i + 1) % 5) for i in range(5)])
K33 = build_graph(range(6), [(i, j) for i in range(3) for j in range(3, 6)])

for G, expected in [(C6, True), (C5, False), (K33, True), (T, True), (G1, False)]:
    c = two_color(G)
    assert (c is not None) == expected == nx.is_bipartite(to_nx(G))
    if c is not None:                       # the coloring must be proper
        assert all(c[u] != c[v] for u in G for v in G[u])
print("Exercise 6 ✔  (even cycles, trees, K33: yes — odd cycles: no)")
draw_graph(C6, coloring=two_color(C6), title="C6 is 2-colorable")
draw_graph(K33, coloring=two_color(K33), title="K(3,3) is bipartite")

## Exercise 7 — Greedy coloring

Implement `greedy_coloring(G, order=None)`: scan the vertices in the given order — by default the *Welsh–Powell* order, i.e. **decreasing degree** — and give each vertex the **smallest** color (a non-negative integer) not already used by its neighbors. Return the dict `{vertex: color}`.

The test cell applies it to wheel graphs and to the **Petersen graph**, and compares the number of colors with networkx's greedy coloring.

**Question.** The wheel $W_n$ is a cycle on $n$ vertices plus a hub joined to every cycle vertex. How many colors are needed when $n$ is even? When $n$ is odd?

In [ ]:
def greedy_coloring(G, order=None):
    """Greedy proper coloring of G following `order` (default: decreasing degree)."""
    # TODO — for each vertex, collect the neighbors' colors, take the smallest free one
    raise NotImplementedError

In [ ]:
def wheel(n):
    """Cycle 0..n-1 plus a hub 'h' joined to every cycle vertex."""
    G = build_graph(range(n), [(i, (i + 1) % n) for i in range(n)])
    for i in range(n):
        add_edge(G, "h", i)
    return G

PETERSEN_EDGES = [(0,1),(1,2),(2,3),(3,4),(4,0),
                  (5,7),(7,9),(9,6),(6,8),(8,5),
                  (0,5),(1,6),(2,7),(3,8),(4,9)]
P = build_graph(range(10), PETERSEN_EDGES)

for G in [wheel(6), wheel(5), P, G1]:
    c = greedy_coloring(G)
    assert all(c[u] != c[v] for u in G for v in G[u])       # proper coloring

n_colors = lambda c: max(c.values()) + 1
print("colors used:  W6 ->", n_colors(greedy_coloring(wheel(6))),
      "| W5 ->", n_colors(greedy_coloring(wheel(5))),
      "| Petersen ->", n_colors(greedy_coloring(P)),
      "| networkx on Petersen ->", n_colors(nx.greedy_color(to_nx(P))))
draw_graph(P, coloring=greedy_coloring(P), title="Petersen graph — greedy coloring")
print("Exercise 7 ✔")

## ★ Challenge — Satisfiability as a coloring problem

**The 3-SAT problem.** A *literal* is a variable `x` or its negation `~x`. A *clause* is an OR of three literals, e.g. $(x \lor \lnot y \lor z)$. A *formula* is an AND of clauses. **Question: is there a True/False assignment of the variables making the whole formula true?**

This is a famously hard decision problem — and it can be **encoded as graph 3-coloring**: from any formula $\varphi$ we build a graph $G_\varphi$ that is 3-colorable **if and only if** $\varphi$ is satisfiable. Translating one problem into another like this is called a *reduction*, and it is the standard tool for comparing the difficulty of problems.

**The construction.** Fix three colors with intended meanings **T**rue, **F**alse, **B**ase:

* a **base triangle** `T – F – B` forces those three vertices to receive three different colors;
* for each variable `x`, a **variable triangle** `x – ~x – B`: both literal vertices must avoid B's color, so each is colored "T" or "F", and `x`, `~x` get **opposite** truth values;
* for each clause, an **OR gadget**. For two inputs `u`, `v`: a fresh triangle `a – b – out`, plus the wires `a – u` and `b – v`. *Key property:* if `u` and `v` are both colored F, then `a` and `b` must use the colors of T and B, forcing `out` to be colored F. If at least one input is colored T, `out` **can** be colored T. (Check both claims on paper!)
* For a 3-literal clause $(\ell_1 \lor \ell_2 \lor \ell_3)$, chain two gadgets — `o1 = OR(l1, l2)`, then `o2 = OR(o1, l3)` — and **force** `o2` to be colored T by joining it to both `F` and `B`.

If every literal of a clause were colored F, its `o2` would be forced to F — impossible. So **any** proper 3-coloring of $G_\varphi$ hands us a satisfying assignment: read off which literals share T's color.

**Your tasks** are marked TODO below: finish the gadget wiring, write the 3-coloring backtracking solver, and decode the assignment.

In [ ]:
def neg(lit):
    """Negation of a literal: 'x' <-> '~x'."""
    return lit[1:] if lit.startswith("~") else "~" + lit

assert neg("x") == "~x" and neg("~x") == "x"

In [ ]:
def build_sat_graph(formula):
    """Graph that is 3-colorable iff `formula` (a list of 3-literal clauses) is satisfiable."""
    G = {}
    # base triangle
    add_edge(G, "T", "F"); add_edge(G, "T", "B"); add_edge(G, "F", "B")

    # one triangle  x -- ~x -- B  per variable
    variables = {lit.lstrip("~") for clause in formula for lit in clause}
    for x in sorted(variables):
        add_edge(G, x, neg(x))
        add_edge(G, x, "B")
        add_edge(G, neg(x), "B")

    def or_gadget(u, v, tag):
        """Create fresh vertices tag+'a', tag+'b', tag+'o' and return tag+'o'.
        Wiring: triangle a-b-o, plus a-u and b-v."""
        # TODO — 5 calls to add_edge, then return the output vertex
        raise NotImplementedError

    for i, (l1, l2, l3) in enumerate(formula):
        # TODO — chain two OR gadgets (tags f"c{i}_1" and f"c{i}_2"),
        #        then force the final output to be colored T
        #        by joining it to "F" and to "B".
        raise NotImplementedError
    return G

In [ ]:
def three_color(G):
    """A proper 3-coloring of G as a dict {vertex: 0, 1 or 2}, or None if impossible.
    Backtracking: order the vertices (decreasing degree works well), then try
    each color in turn for the current vertex and recurse."""
    nodes = sorted(G, key=lambda v: -len(G[v]))
    color = {}

    def backtrack(i):
        # TODO — if i == len(nodes): done. Otherwise try each color c in 0,1,2
        #        that no already-colored neighbor of nodes[i] uses.
        raise NotImplementedError

    return color if backtrack(0) else None

In [ ]:
def decode(coloring, formula):
    """Read the truth assignment off a 3-coloring of the SAT graph:
    a variable is True iff its vertex has the same color as vertex 'T'."""
    # TODO — return a dict {variable: bool}
    raise NotImplementedError

def evaluate(formula, assignment):
    """Truth value of the formula under the assignment (provided)."""
    def value(lit):
        return not assignment[lit[1:]] if lit.startswith("~") else assignment[lit]
    return all(any(value(lit) for lit in clause) for clause in formula)

In [ ]:
PHI = [("x", "y", "z"),
       ("~x", "~y", "z"),
       ("x", "~y", "~z"),
       ("~x", "y", "~z")]

G_sat = build_sat_graph(PHI)
print("SAT graph:", len(G_sat), "vertices,", num_edges(G_sat), "edges")

col = three_color(G_sat)
assert col is not None, "PHI is satisfiable, so a 3-coloring must exist"
assert all(col[u] != col[v] for u in G_sat for v in G_sat[u])   # proper

assignment = decode(col, PHI)
print("assignment read off the coloring:", assignment)
assert evaluate(PHI, assignment), "the decoded assignment must satisfy PHI"
print("Challenge ✔ — the coloring solved the formula!")
draw_graph(G_sat, coloring=col, title="The 3-SAT graph of PHI, properly 3-colored")

**Going further (optional).** The formula below contains all $8$ possible clauses on `x, y, z`, so *every* assignment falsifies one of them: it is **unsatisfiable**, and its graph should therefore **not** be 3-colorable. Uncomment and run — the backtracking has to exhaust the search space, so it takes noticeably longer than the satisfiable case. That asymmetry (easy to verify a solution, hard to rule one out) is exactly what makes such problems interesting.

In [ ]:
UNSAT = [(a, b, c) for a in ("x", "~x") for b in ("y", "~y") for c in ("z", "~z")]
# print(three_color(build_sat_graph(UNSAT)))     # expected: None

## Summary

You implemented from scratch, on dict-of-lists graphs: construction, degrees (+ handshake lemma), BFS distances, DFS, connected components, cycle detection, the bipartite test by 2-coloring, greedy coloring — and used 3-coloring to *solve logic formulas* through the 3-SAT reduction. Throughout, `networkx` served as an independent referee for your code.